In [1]:
import folium

# Pusat peta di tengah area AOI
lat_c = (-7.1927923 + -7.1514786) / 2
lon_c = (112.6193968 + 112.6600158) / 2

m = folium.Map(location=[lat_c, lon_c], zoom_start=12)

# Tandai area polygon AOI Kabupaten Gresik
folium.Rectangle(
    bounds=[[-7.1927923, 112.6193968], [-7.1514786, 112.6600158]],
    color="red",
    fill=True,
    fill_opacity=0.2,
    tooltip="Area Kabupaten Gresik",
).add_to(m)

m

In [2]:
import pandas as pd

# Menampilkan 5 data teratas CSV CH4
df_ch4 = pd.read_csv("./../data/csv/CH4_gresik_timeseries.csv")
df_ch4.head()

,date,CH4
0,2025-08-24,NaN
1,2025-08-25,1867.289429
2,2025-08-26,NaN
3,2025-08-27,NaN
4,2025-08-28,NaN


In [3]:
import pandas as pd

# Menampilkan 5 data teratas CSV CO
df_co = pd.read_csv("./../data/csv/CO_gresik_timeseries.csv")
df_co.head()

,date,CO
0,2025-08-24,0.037918
1,2025-08-25,0.028240
2,2025-08-26,NaN
3,2025-08-27,0.034187
4,2025-08-28,0.027450


In [4]:
import pandas as pd

# Menampilkan 5 data teratas CSV NO2
df_no2 = pd.read_csv("./../data/csv/NO2_gresik_timeseries.csv")
df_no2.head()

,date,NO2
0,2025-08-24,0.000096
1,2025-08-25,0.000046
2,2025-08-26,0.000068
3,2025-08-27,NaN
4,2025-08-28,0.000060


In [5]:
import pandas as pd

# Menampilkan 5 data teratas CSV SO2
df_so2 = pd.read_csv("./../data/csv/SO2_gresik_timeseries.csv")
df_so2.head()

,date,SO2
0,2025-08-24,0.000347
1,2025-08-25,0.000817
2,2025-08-26,-0.000799
3,2025-08-27,-0.000513
4,2025-08-28,0.000327


In [6]:
import pandas as pd

polutans = ["NO2", "CO", "SO2", "CH4"]
data = []

for pol in polutans:
    df = pd.read_csv(f"./../data/csv/{pol}_gresik_timeseries.csv")
    total = len(df)
    missing = int(df[pol].isna().sum())
    data.append({"Polutan": pol, "Total": total, "Ada Data": total - missing,
                 "Missing (NaN)": missing, "Persentase Missing": f"{missing/total*100:.1f}%"})

pd.DataFrame(data)

,Polutan,Total,Ada Data,Missing (NaN),Persentase Missing
0,NO2,365,187,178,48.8%
1,CO,365,209,156,42.7%
2,SO2,365,222,143,39.2%
3,CH4,365,31,334,91.5%


In [7]:
import pandas as pd

polutans = ["NO2", "CO", "SO2", "CH4"]
data = []

for pol in polutans:
    df = pd.read_csv(f"./../data/csv/{pol}_gresik_timeseries.csv")
    s = df[pol].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    n_out = int(((s < lo) | (s > hi)).sum())
    data.append({"Polutan": pol, "Q1": round(q1, 6), "Q3": round(q3, 6),
                 "IQR": round(iqr, 6), "Batas Bawah": round(lo, 6),
                 "Batas Atas": round(hi, 6), "Jumlah Outlier": n_out})

pd.DataFrame(data)

,Polutan,Q1,Q3,IQR,Batas Bawah,Batas Atas,Jumlah Outlier
0,NO2,0.000044,0.000079,0.000035,-0.000009,0.000131,12
1,CO,0.026579,0.031304,0.004725,0.019491,0.038392,7
2,SO2,-0.000075,0.000314,0.000389,-0.000659,0.000897,3
3,CH4,1886.370789,1905.616455,19.245667,1857.502289,1934.484955,1


In [8]:
import pandas as pd

polutans = ["NO2", "CO", "SO2", "CH4"]
data = []

for pol in polutans:
    df = pd.read_csv(f"./../data/csv/{pol}_gresik_timeseries.csv")
    s = df[pol].dropna()
    diff = s.diff().abs()
    threshold = 3 * s.std()
    n_noise = int((diff > threshold).sum())
    data.append({"Polutan": pol, "Ambang (3×SD)": round(threshold, 6),
                 "Jumlah Kandidat Noise": n_noise})

pd.DataFrame(data)

,Polutan,Ambang (3×SD),Jumlah Kandidat Noise
0,NO2,0.000190,6
1,CO,0.012793,5
2,SO2,0.000960,7
3,CH4,52.090372,0
